In [1]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory: {total:.1f} GB")

Sat Sep 26 12:56:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U peft bitsandbytes

import transformers, peft, bitsandbytes
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.6 MB/s eta 0:00:00
transformers: 5.16.1
peft: 0.21.0
bitsandbytes: 0.50.2


In [4]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                     # store weights in 4 bits
    bnb_4bit_quant_type="nf4",             # the 4-bit format QLoRA uses
    bnb_4bit_compute_dtype=torch.float16,  # T4 does not support bfloat16, so use float16
    bnb_4bit_use_double_quant=True,        # saves a little more memory
)

torch.cuda.reset_peak_memory_stats()
start = time.time()

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"Load time: {time.time() - start:.1f} s")
print(f"GPU memory after loading: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print(f"Peak GPU memory while loading: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print("Tokenizer vocab size:", len(tokenizer))
print("Model embedding size:", model.config.vocab_size)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Load time: 56.0 s
GPU memory after loading: 1.07 GB
Peak GPU memory while loading: 1.10 GB
Tokenizer vocab size: 151665
Model embedding size: 151936


In [3]:
messages = [
    {"role": "user", "content": "Rewrite this Banglish sentence in clear English: amar bkash e 5oo tk pathaisi kintu pay nai"}
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

start = time.time()
out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"Generation time: {time.time() - start:.1f} s")

NameError: name 'tokenizer' is not defined

In [6]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Load only if not already loaded in this session
if "model" not in globals() or "tokenizer" not in globals():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto"
    )
    print("Model loaded.")
else:
    print("Model already loaded, reusing it.")

print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Quick test on a Banglish sentence
messages = [
    {"role": "user", "content": "Rewrite this Banglish sentence in clear English: amar bkash e 5oo tk pathaisi kintu pay nai"}
]
inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)

start = time.time()
out = model.generate(**inputs, max_new_tokens=60, do_sample=False)
print("MODEL OUTPUT:", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))
print(f"Generation time: {time.time() - start:.1f} s")

Model already loaded, reusing it.
GPU memory in use: 1.08 GB
MODEL OUTPUT: The sentence "amar bkash e 5oo tk pathaisi kintu pay nai" is written in Bengali and translates to:

"I have sent 500 Tk via Bkash."

This means that the person has used their Bkash account to send money (in
Generation time: 6.2 s


In [7]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

if not hasattr(model, "peft_config"):          # don't add adapters twice
    model = prepare_model_for_kbit_training(model)   # makes 4-bit training stable, saves memory
    lora_config = LoraConfig(
        r=16,               # adapter size: bigger = more capacity, more memory
        lora_alpha=32,      # how strongly the adapters affect the output
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    print("LoRA adapters added.")
else:
    print("LoRA adapters already present.")

model.print_trainable_parameters()
print(f"GPU memory in use: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

LoRA adapters added.
trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820
GPU memory in use: 1.59 GB


In [8]:
import json, time, torch

SYSTEM = ("You sanitize Romanized Bangla (Banglish) prompts. Return only JSON with keys: "
          "sanitized_prompt, pii, preserved_entities, uncertainties, routing.")
example_input = "kalk3 bkash e 5oo tk send korsi trans id 9X87K, taka pay nai. 1987654321 eta check koren"
example_target = {
    "sanitized_prompt": "Yesterday I sent 500 BDT via bKash (transaction ID <TXN_ID_1>), but the money was not received. Please check <ID_NUMBER_1>.",
    "pii": [{"type": "TXN_ID", "placeholder": "<TXN_ID_1>"},
            {"type": "ID_NUMBER", "placeholder": "<ID_NUMBER_1>"}],
    "preserved_entities": [{"type": "AMOUNT", "value": "500 BDT"},
                           {"type": "SERVICE", "value": "bKash"}],
    "uncertainties": [
        {"span": "5oo tk", "types": ["NUMERIC_AMBIGUITY"], "candidates": ["500 BDT", "5000 BDT"],
         "aleatoric": 0.21, "epistemic": 0.63},
        {"span": "1987654321", "types": ["PII_BOUNDARY"], "candidates": ["NID", "account number"],
         "aleatoric": 0.74, "epistemic": 0.12}],
    "routing": "ESCALATE",
}

# Build token ids: prompt part + target part
prompt_text = tokenizer.apply_chat_template(
    [{"role": "system", "content": SYSTEM}, {"role": "user", "content": example_input}],
    add_generation_prompt=True, tokenize=False)
prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
target_ids = tokenizer(json.dumps(example_target, ensure_ascii=False) + tokenizer.eos_token,
                       add_special_tokens=False)["input_ids"]

ids = prompt_ids + target_ids
labels = [-100] * len(prompt_ids) + target_ids      # learn only the target
print(f"Prompt tokens: {len(prompt_ids)}, target tokens: {len(target_ids)}, total: {len(ids)}")

BATCH = 4
input_ids = torch.tensor([ids] * BATCH, device=model.device)
label_ids = torch.tensor([labels] * BATCH, device=model.device)
attention = torch.ones_like(input_ids)

# Training loop
STEPS = 30
model.train()
model.config.use_cache = False
optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-4)

torch.cuda.reset_peak_memory_stats()
times = []
for step in range(1, STEPS + 1):
    t = time.time()
    loss = model(input_ids=input_ids, attention_mask=attention, labels=label_ids).loss
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    torch.cuda.synchronize()
    times.append(time.time() - t)
    if step in (1, 5, 10, 20, 30):
        print(f"step {step:2d}  loss {loss.item():.4f}  time {times[-1]:.2f} s")

avg = sum(times[1:]) / len(times[1:])   # skip step 1 (warm-up is slower)
print(f"\nPeak GPU memory during training: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")
print(f"Average time per step (batch {BATCH}): {avg:.2f} s")

Prompt tokens: 86, target tokens: 253, total: 339


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


step  1  loss 1.3672  time 2.75 s
step  5  loss 0.3402  time 2.07 s
step 10  loss 0.0214  time 2.16 s
step 20  loss 0.0004  time 2.15 s
step 30  loss 0.0001  time 2.02 s

Peak GPU memory during training: 4.26 GB
Average time per step (batch 4): 2.14 s


In [9]:
model.eval()
model.config.use_cache = True

inputs = tokenizer(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
start = time.time()
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(f"Generation time: {time.time() - start:.1f} s\n")
print(text)

try:
    parsed = json.loads(text)
    print("\nVALID JSON. Keys:", list(parsed.keys()))
except json.JSONDecodeError as e:
    print("\nNOT valid JSON:", e)

Generation time: 36.9 s

{"sanitized_prompt": "Yesterday I sent 500 BDT via bKash (transaction ID <TXN_ID_1>), but the money was not received. Please check <ID_NUMBER_1>.", "pii": [{"type": "TXN_ID", "placeholder": "<TXN_ID_1>"}, {"type": "ID_NUMBER", "placeholder": "<ID_NUMBER_1>"}], "preserved_entities": [{"type": "AMOUNT", "value": "500 BDT"}, {"type": "SERVICE", "value": "bKash"}], "uncertainties": [{"span": "5oo tk", "types": ["NUMERIC_AMBIGUITY"], "candidates": ["500 BDT", "5000 BDT"], "aleatoric": 0.21, "epistemic": 0.63}, {"span": "1987654321", "types": ["PII_BOUNDARY"], "candidates": ["NID", "account number"], "aleatoric": 0.74, "epistemic": 0.12}], "routing": "ESCALATE"}

VALID JSON. Keys: ['sanitized_prompt', 'pii', 'preserved_entities', 'uncertainties', 'routing']
